# Latvia Temperature Patterns — Public Weather Data

## Project overview

This case study reorganises an earlier Python exercise built on public Latvian open-data weather information. The original temperature dataset contained **599,640 observations** across **5,260 latitude/longitude pairs** for the period 13–22 November 2023.

To keep the GitHub repository lightweight, this portfolio version includes:

- a daily summary calculated from the full source dataset; and
- a deterministic sample of 40 locations for location-level exploration.

The original exercise referenced Latvia's open-data portal (`data.gov.lv`) as the source.

### Analytical questions

- How did daily temperature change during the observation period?
- What were the warmest and coldest daily values?
- How much spatial variation is visible across the sampled locations?

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DAILY_PATH = Path("../data/latvia_temperature_daily_summary.csv")
SAMPLE_PATH = Path("../data/latvia_temperature_location_sample.csv")

daily = pd.read_csv(DAILY_PATH, parse_dates=["date"])
sample = pd.read_csv(SAMPLE_PATH, parse_dates=["Datetime"])

daily.head(), sample.head()

## Data overview

In [ ]:
print(f"Daily summary period: {daily['date'].min().date()} to {daily['date'].max().date()}")
print(f"Sample rows: {len(sample):,}")
print(f"Sample locations: {sample[['Lat', 'Lon']].drop_duplicates().shape[0]}")
print(f"Sample timestamps: {sample['Datetime'].nunique()}")

sample[["Value"]].describe().round(2)

## Daily temperature range from the full dataset

In [ ]:
daily.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.plot(daily["date"], daily["mean_temp_c"], marker="o", label="Mean")
ax.fill_between(
    daily["date"],
    daily["min_temp_c"],
    daily["max_temp_c"],
    alpha=0.2,
    label="Daily min-max range"
)
ax.axhline(0, linewidth=1)
ax.set_title("Daily temperature summary — 13 to 22 November 2023")
ax.set_xlabel("Date")
ax.set_ylabel("Temperature (°C)")
ax.tick_params(axis="x", rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

The period becomes progressively colder overall. The full dataset's daily mean falls from above 3 °C on 13 November to below -4 °C on 22 November, although there are short warmer intervals in between.

## Coldest and warmest days

In [ ]:
coldest = daily.loc[daily["min_temp_c"].idxmin()]
warmest = daily.loc[daily["max_temp_c"].idxmax()]

print(
    f"Coldest observed daily minimum: {coldest['min_temp_c']:.1f} °C "
    f"on {coldest['date'].date()}"
)
print(
    f"Warmest observed daily maximum: {warmest['max_temp_c']:.1f} °C "
    f"on {warmest['date'].date()}"
)

## Spatial variation in the location sample

In [ ]:
location_summary = (
    sample.groupby(["Lat", "Lon"])["Value"]
          .agg(mean_temp_c="mean", min_temp_c="min", max_temp_c="max", observations="size")
          .reset_index()
)

location_summary.sort_values("mean_temp_c").head()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(
    location_summary["Lon"],
    location_summary["Lat"],
    c=location_summary["mean_temp_c"],
    s=70
)
ax.set_title("Mean temperature across sampled locations")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.colorbar(sc, ax=ax, label="Mean temperature (°C)")
plt.tight_layout()
plt.show()

## Hourly pattern across sampled locations

In [ ]:
hourly_sample = (
    sample.groupby("Datetime")["Value"]
          .mean()
          .reset_index(name="mean_temp_c")
)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(hourly_sample["Datetime"], hourly_sample["mean_temp_c"])
ax.axhline(0, linewidth=1)
ax.set_title("Mean temperature over time across sampled locations")
ax.set_xlabel("Datetime")
ax.set_ylabel("Mean temperature (°C)")
plt.tight_layout()
plt.show()

## Key findings

- The observation period shows a clear transition toward colder conditions.
- In the full dataset, the minimum temperature reached **-9.7 °C** on 22 November, while the highest daily maximum reached **7.6 °C** on 20 November.
- Temperature varies across locations as well as over time, which is visible in the sampled coordinate-level data.
- The portfolio version separates full-dataset daily summaries from a smaller spatial sample to keep the repository reproducible and compact.

## Limitations

The project analyses a short ten-day period and should not be interpreted as a climatological assessment. The included location sample is intended for demonstration of spatial exploration; it is not a substitute for the complete raw dataset.